In [3]:
import pandas as pd


df = pd.read_csv('student-mat.csv', sep=';')
print("Original shape:", df.shape)


df['failed'] = (df['G3'] < 10).astype(int)
print("Fail rate:", round(df['failed'].mean(), 3))


df_model = df.drop(columns=['G1', 'G2', 'G3', 'failures'])
print("Number of columns after dropping leakage features:", df_model.shape[1])
print("Column names:", list(df_model.columns))

Original shape: (395, 33)
Fail rate: 0.329
Number of columns after dropping leakage features: 30
Column names: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'failed']


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


X = df_model.drop(columns=['failed'])
y = df_model['failed']


X_encoded = X.copy()
categorical_cols = X.select_dtypes(include=['object']).columns

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])

print("Categorical columns encoded:", list(categorical_cols))


X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Train fail rate:", round(y_train.mean(), 3))
print("Test fail rate:", round(y_test.mean(), 3))

Categorical columns encoded: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Train size: 276
Test size: 119
Train fail rate: 0.33
Test fail rate: 0.328


In [6]:
from sklearn.tree import DecisionTreeClassifier


model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)

print("Decision tree trained successfully.")
print("Max depth used:", model.get_depth())
print("Number of leaves:", model.get_n_leaves())

Decision tree trained successfully.
Max depth used: 5
Number of leaves: 19


In [7]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score


y_pred = model.predict(X_test)


cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("(Rows = Actual, Columns = Predicted)")
print("Order: [Pass, Fail]")


print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Pass', 'Fail']))

print("Overall Accuracy:", round(accuracy_score(y_test, y_pred), 3))

Confusion Matrix:
[[67 13]
 [29 10]]
(Rows = Actual, Columns = Predicted)
Order: [Pass, Fail]

Classification Report:
              precision    recall  f1-score   support

        Pass       0.70      0.84      0.76        80
        Fail       0.43      0.26      0.32        39

    accuracy                           0.65       119
   macro avg       0.57      0.55      0.54       119
weighted avg       0.61      0.65      0.62       119

Overall Accuracy: 0.647


In [10]:

test_results = X_test.copy()
test_results['actual'] = y_test.values
test_results['predicted'] = y_pred


false_negatives = test_results[(test_results['actual'] == 1) & (test_results['predicted'] == 0)]


false_positives = test_results[(test_results['actual'] == 0) & (test_results['predicted'] == 1)]

print("Number of False Negatives (missed fails):", len(false_negatives))
print("Number of False Positives:", len(false_positives))

print("\nFalse Negatives average values:")
print("studytime:", round(false_negatives['studytime'].mean(), 2))
print("absences:", round(false_negatives['absences'].mean(), 2))
print("Medu:", round(false_negatives['Medu'].mean(), 2))
print("goout:", round(false_negatives['goout'].mean(), 2))
print("age:", round(false_negatives['age'].mean(), 2))

Number of False Negatives (missed fails): 29
Number of False Positives: 13

False Negatives average values:
studytime: 1.86
absences: 6.55
Medu: 2.59
goout: 3.21
age: 16.9
